In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 300

# Generate Synthetic Mystery Ops Dataset (Oil & Gas Pipeline Telemetry)
np.random.seed(42)
n_rows = 2000

timestamps = pd.date_range(start="2026-05-01", periods=n_rows, freq="H")
stations = np.choice(
    ["Station_Alpha", "Station_Beta", "Station_Gamma", "Station_Delta"],
    size=n_rows,
)
base_pressure = np.random.normal(120.0, 5.0, n_rows)
base_flow = np.random.normal(450.0, 25.0, n_rows)
temperatures = np.random.normal(28.0, 4.0, n_rows)

# Inject anomalies into Station_Gamma
gamma_mask = stations == "Station_Gamma"
base_pressure[gamma_mask] -= np.random.uniform(
    15, 35, size=gamma_mask.sum()
)  # Pressure drops
base_flow[gamma_mask] -= np.random.uniform(
    50, 120, size=gamma_mask.sum()
)  # Flow drops

df = pd.DataFrame(
    {
        "Timestamp": timestamps,
        "Station_ID": stations,
        "Pressure_PSI": base_pressure,
        "Flow_Rate_BPH": base_flow,
        "Ambient_Temp_C": temperatures,
        "Valve_Status": np.random.choice(
            ["Optimal", "Warning", "Critical"],
            size=n_rows,
            p=[0.8, 0.15, 0.05],
        ),
    }
)

# Introduce minor missing values for profiling demonstration
df.loc[np.random.choice(n_rows, 15), "Pressure_PSI"] = np.nan
df.head()

: 

In [ ]:
# Summary statistics and missing value check
print("--- DATA SUMMARY STATISTICS ---")
display(df.describe())

print("\n--- MISSING VALUES CHECK ---")
print(df.isnull().sum())

# Distribution Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    df["Pressure_PSI"],
    kde=True,
    ax=axes[0],
    color="#1f77b4",
    bins=30,
)
axes[0].set_title(
    "Univariate Distribution: Pipeline Pressure (PSI)",
    fontsize=12,
    fontweight="bold",
)
axes[0].set_xlabel("Pressure (PSI)")

sns.histplot(
    df["Flow_Rate_BPH"],
    kde=True,
    ax=axes[1],
    color="#ff7f0e",
    bins=30,
)
axes[1].set_title(
    "Univariate Distribution: Flow Rate (BPH)", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Flow Rate (BPH)")

plt.tight_layout()
plt.show()

In [ ]:
# Identify outliers using Interquartile Range (IQR) for Pressure
Q1 = df["Pressure_PSI"].quantile(0.25)
Q3 = df["Pressure_PSI"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR

anomalies = df[df["Pressure_PSI"] < lower_bound]
print(f"Total detected pressure anomalies: {len(anomalies)}")

# Visualization: Box Plot & Scatter Plot highlighting anomalies
plt.figure(figsize=(12, 6))
sns.boxplot(
    x="Station_ID",
    y="Pressure_PSI",
    data=df,
    palette="Set2",
    hue="Station_ID",
    legend=False,
)
plt.title(
    "Anomaly Detection: Pressure Distribution Across Stations (Box Plot)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Station ID")
plt.ylabel("Pressure (PSI)")
plt.axhline(
    lower_bound,
    color="red",
    linestyle="--",
    label=f"Outlier Threshold ({lower_bound:.1f} PSI)",
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Drill-down by Station
station_summary = (
    df.groupby("Station_ID")[["Pressure_PSI", "Flow_Rate_BPH"]]
    .mean()
    .reset_index()
)
display(station_summary)

# 2. Correlation Analysis
plt.figure(figsize=(8, 6))
corr = df[
    ["Pressure_PSI", "Flow_Rate_BPH", "Ambient_Temp_C"]
].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title(
    "Correlation Matrix: Operational Parameters",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# 3. Pareto Analysis on Valve Status / Downtime causes
valve_counts = (
    df[df["Valve_Status"] != "Optimal"]["Station_ID"]
    .value_counts()
    .reset_index()
)
valve_counts.columns = ["Station_ID", "Issue_Count"]
valve_counts["Cumulative_Percentage"] = (
    valve_counts["Issue_Count"].cumsum() / valve_counts["Issue_Count"].sum()
) * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(
    valve_counts["Station_ID"],
    valve_counts["Issue_Count"],
    color="#2ca02c",
    alpha=0.8,
)
ax1.set_ylabel("Critical / Warning Incidents", color="#2ca02c", fontweight="bold")
ax1.tick_params(axis="y", labelcolor="#2ca02c")

ax2 = ax1.twinx()
ax2.plot(
    valve_counts["Station_ID"],
    valve_counts["Cumulative_Percentage"],
    color="#d62728",
    marker="o",
    linewidth=2,
)
ax2.set_ylabel(
    "Cumulative Percentage (%)", color="#d62728", fontweight="bold"
)
ax2.set_ylim(0, 105)

plt.title(
    "Pareto Analysis: Incident Concentration by Station",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()